# H&M 거래·상품 이미지 표본 추출

비공개 Kaggle Notebook에서 **Add Input**으로 H&M Personalized Fashion Recommendations 대회를 연결한 뒤 실행한다. 대회 입력은 Kaggle에서 읽고, 로컬로 옮길 파일은 이용 조건을 확인한 후 `hm_sample.zip` 하나만 선택한다. 표본 ZIP과 해제한 파일은 로컬 프로젝트의 `data/raw/hm/`에 둔다.

기본 기간은 2020년 6~8월이며 원본이 이 기간을 포함하는지 실행 중 검사한다. 초깃값은 고객 최대 1,000명과 실제 상품 이미지 최대 200장이다. 채널 코드가 확인될 때까지 두 채널을 함께 사용한다. 구매일과 고객 ID, 원래 가격을 보존하며, RFM 기준일은 표본 추출 전 범위에서 결정한다.

### 1. 환경 경로 및 표본 추출 설정 (`In [1]`)

* **입출력 경로 지정:** Kaggle 대회 원본 데이터 경로(`/kaggle/input`)와 추출된 압축 파일이 저장될 작업 경로(`/kaggle/working`)를 설정합니다.
* **추출 조건 및 시드 고정:**
  * **기간(`START_DATE` ~ `END_DATE`):** 2020년 6월 1일부터 8월 31일까지(3개월)의 거래 기록을 대상으로 합니다.
  * **표본 규모:** 재현성을 위해 난수 시드(`SEED = 42`)를 고정하고, 최대 고객 수 **1,000명**, 상품 이미지 **200장**, 최소 거래 건수 **1,000건**을 기준으로 설정합니다.
* **청크 크기(`CHUNK_SIZE`):** 약 3,178만 행에 달하는 대용량 원본 CSV를 메모리 초과 없이 처리하기 위해 **25만 행** 단위로 분할하여 읽습니다.

In [ ]:
from pathlib import Path	# 경로 처리용 표준 라이브러리

INPUT_ROOT = Path("/kaggle/input")	# Kaggle 입력 데이터 루트 경로
DATASET_DIR = None	# 후보가 여러 개일 때 직접 지정할 데이터셋 폴더 경로
OUTPUT_DIR = Path("/kaggle/working")	# 추출 결과 ZIP 저장 경로

START_DATE = "2020-06-01"	# 표본 추출 시작일(포함)
END_DATE = "2020-09-01"	# 표본 추출 종료일(미포함)
SALES_CHANNEL = None	# 채널 의미 확인 전까지 전체 채널 포함
SEED = 42	# 무작위 추출 재현용 난수 시드
MAX_CUSTOMERS = 1_000	# 추출할 최대 고객 수
MAX_IMAGES = 200	# 추출할 최대 상품 이미지 수
MIN_TRANSACTIONS = 1_000	# 최소 유효 거래 행 수 기준
CHUNK_SIZE = 250_000	# 대용량 CSV 분할 읽기 단위(행)


### 2. 대용량 데이터 청크 처리 및 표본 추출 함수 정의 (`In [2]`)

* **`find_dataset()`:** `/kaggle/input` 하위 디렉터리에서 `transactions_train.csv`, `articles.csv`, `images/` 폴더가 모두 포함된 H&M 원본 데이터셋 위치를 자동으로 탐색합니다.
* **`transaction_chunks()`:** 원본 거래 파일을 25만 행씩 순회하며 결측 고객 ID, 잘못된 날짜 형식, `0` 이하의 비정상 가격 행을 상호 배타적으로 집계·제외하고 지정 기간(`2020-06-01` ~ `2020-08-31`) 내 유효 거래만 필터링합니다.
* **`choose()`:** 정렬된 고유 ID 목록에서 고정 시드(`np.random.default_rng`) 기반 비복원 무작위 추출을 수행해 실행 환경에 관계없이 동일한 표본을 보장합니다.
* **`prepare_hm_sample()`:**
  1. **1차 스캔:** 전체 기간의 유효 고객 집합과 RFM 분석 기준일(`reference_date = 2020-08-31`)을 산출한 뒤, 최대 1,000명의 표본 고객을 선정합니다.
  2. **2차 스캔 및 메타데이터 결합:** 선정된 고객의 거래 내역(`transactions_sample.csv`)을 추출하고, 연결된 상품 메타데이터(`articles_sample.csv`)와 최대 200장의 상품 이미지를 매칭합니다.
  3. **검증 및 패키징:** 제외된 행 수와 추출 행 수의 합계가 원본 전체 행 수와 일치하는지 정합성을 검증한 후, 감사 로그(`sample_manifest.json`)와 함께 `hm_sample.zip`으로 압축합니다.

In [ ]:
import json	# JSON 직렬화 및 저장용 모듈
from collections import Counter	# 제외·추출 건수 누적 집계용
from tempfile import TemporaryDirectory	# 작업 중 임시 폴더 생성용
from zipfile import ZIP_DEFLATED, ZIP_STORED, ZipFile	# ZIP 압축 방식 및 아카이브 생성용

import numpy as np	# 수치 연산 및 난수 시드 추출용
import pandas as pd	# CSV 청크 읽기 및 데이터 정제용


TRANSACTION_COLUMNS = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]	# 거래 CSV에서 읽을 필수 열 목록
ARTICLE_COLUMNS = ["article_id", "prod_name", "product_type_name"]	# 상품 CSV에서 읽을 필수 열 목록


def find_dataset(input_root):
    """입력 루트에서 H&M 거래·상품·이미지 폴더를 모두 갖춘 단일 데이터셋 경로를 찾는다.

    Args:
        input_root (Path): Kaggle 입력 루트 디렉터리.

    Returns:
        Path: transactions_train.csv, articles.csv, images/가 모두 있는 데이터셋 디렉터리.
    """
    candidates = [	# 깊이 0~2 범위에서 데이터셋 후보 디렉터리 탐색
        path.parent	# transactions_train.csv가 위치한 부모 폴더
        for pattern in ("transactions_train.csv", "*/transactions_train.csv", "*/*/transactions_train.csv")	# 탐색할 glob 패턴 목록
        for path in input_root.glob(pattern)	# 패턴에 맞는 거래 CSV 경로 순회
        if (path.parent / "articles.csv").is_file() and (path.parent / "images").is_dir()	# 상품 CSV와 이미지 폴더 존재 여부 확인
    ]
    if len(candidates) != 1:	# 후보가 없거나 여러 개면 모호하므로 중단
        raise ValueError("Attach the H&M competition input or set DATASET_DIR to its folder.")	# 입력 연결 또는 수동 경로 지정 안내 예외
    return candidates[0]	# 유일하게 확인된 데이터셋 경로 반환


def transaction_chunks(path, start, end, sales_channel, chunk_size):
    """거래 CSV를 청크 단위로 읽어 정제·기간 필터링된 행과 제외 건수 감사를 순차 반환한다.

    Args:
        path (Path): transactions_train.csv 경로.
        start (pd.Timestamp): 포함할 시작 날짜.
        end (pd.Timestamp): 제외할 종료 날짜.
        sales_channel (int | str | None): 필터링할 판매 채널 ID(None이면 전체).
        chunk_size (int): 한 번에 읽을 행 수.

    Yields:
        tuple: (선택된 거래 DataFrame, 날짜 Series, 청크 최소일, 청크 최대일, 제외 건수 dict).
    """
    for rows in pd.read_csv(path, usecols=TRANSACTION_COLUMNS, dtype="string", chunksize=chunk_size):	# 필수 열만 문자열로 청크 읽기
        rows["customer_id"] = rows["customer_id"].str.strip().replace("", pd.NA)	# 고객 ID 공백 제거 및 빈 문자열 결측 처리
        rows["article_id"] = rows["article_id"].str.strip().replace("", pd.NA)	# 상품 ID 공백 제거 및 빈 문자열 결측 처리
        dates = pd.to_datetime(rows["t_dat"], format="%Y-%m-%d", errors="coerce")	# 거래일 문자열을 날짜형으로 변환(오류는 NaT)
        amounts = pd.to_numeric(rows["price"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)	# 금액을 실수 배열로 변환(오류는 NaN)
        customer_ok = rows["customer_id"].notna()	# 유효 고객 ID 마스크
        date_ok = dates.notna()	# 유효 거래일 마스크
        amount_ok = np.isfinite(amounts) & (amounts > 0)	# 유한한 양수 금액 마스크
        valid = customer_ok & date_ok & amount_ok	# 세 기준을 모두 만족하는 유효 거래 마스크
        in_scope = dates.ge(start) & dates.lt(end)	# 지정 기간 [start, end) 내 거래 마스크
        if sales_channel is not None:	# 특정 판매 채널만 추출할 때
            in_scope &= rows["sales_channel_id"].eq(str(sales_channel)).fillna(False)	# 채널 일치 여부 추가 필터링
        audit = {	# 상호 배타적 제외 사유별 건수 기록
            "source_rows": len(rows),	# 현재 청크의 원본 행 수
            "missing_customer_rows": int((~customer_ok).sum()),	# 고객 ID 누락 건수
            "invalid_date_rows": int((customer_ok & ~date_ok).sum()),	# 고객은 유효하나 날짜 오류 건수
            "invalid_amount_rows": int((customer_ok & date_ok & ~amount_ok).sum()),	# 고객·날짜 유효하나 금액 오류 건수
            "out_of_scope_rows": int((valid & ~in_scope).sum()),	# 유효하나 기간·채널 범위 밖 건수
        }
        selected = valid & in_scope	# 유효하면서 범위 내인 최종 선택 마스크
        yield rows.loc[selected, TRANSACTION_COLUMNS], dates.loc[selected], dates.min(), dates.max(), audit	# 필터링 결과와 청크 통계 반환


def choose(values, limit, seed):
    """정렬된 고유 값에서 고정 시드로 중복 없이 최대 limit개를 무작위 추출한다.

    Args:
        values (Iterable): 추출 대상 고유 값 컬렉션.
        limit (int): 최대 추출 개수.
        seed (int): 난수 시드.

    Returns:
        set: 선택된 고유 값 집합.
    """
    ordered = sorted(values)	# 집합 순회 순서 영향 제거를 위한 사전 정렬
    indices = np.random.default_rng(seed).choice(len(ordered), size=min(limit, len(ordered)), replace=False)	# 고정 시드 비복원 인덱스 추출
    return {ordered[index] for index in indices}	# 선택된 인덱스에 해당하는 값 집합 반환


def prepare_hm_sample(dataset_dir, output_dir, *, start, end, seed, max_customers,
                      max_images, min_transactions, chunk_size, sales_channel=None):
    """H&M 원본 데이터에서 표본 거래·상품·이미지를 추출해 검증 후 hm_sample.zip으로 저장한다.

    Args:
        dataset_dir (Path): 원본 데이터셋 디렉터리.
        output_dir (Path): 출력 ZIP을 저장할 디렉터리.
        start (str | pd.Timestamp): 포함할 시작 월초 날짜.
        end (str | pd.Timestamp): 제외할 종료 월초 날짜.
        seed (int): 표본 추출용 난수 시드.
        max_customers (int): 최대 표본 고객 수.
        max_images (int): 최대 표본 상품 이미지 수.
        min_transactions (int): 최소 유효 표본 거래 행 수.
        chunk_size (int): CSV 청크 읽기 크기.
        sales_channel (int | str | None): 선택적 판매 채널 필터.

    Returns:
        tuple[Path, dict]: (생성된 ZIP 경로, 추출 감사 기록 manifest dict).
    """
    start, end = pd.Timestamp(start), pd.Timestamp(end)	# 시작·종료일을 Timestamp로 정규화
    if not (start.is_month_start and end.is_month_start and start + pd.DateOffset(months=3) <= end):	# 최소 3개의 완전한 월 구간인지 검사
        raise ValueError("Use month starts for an interval of at least three complete months.")	# 월초 기준 및 3개월 이상 조건 위반 예외
    if min(max_customers, max_images, min_transactions, chunk_size) < 1:	# 양수 파라미터 여부 검사
        raise ValueError("Sampling limits and chunk size must be positive.")	# 1 미만 설정 시 예외 발생

    transaction_path = dataset_dir / "transactions_train.csv"	# 원본 거래 CSV 경로 지정
    customers, covered_months = set(), set()	# 유효 고객 집합과 관측 월 집합 초기화
    counts = Counter()	# 전체 스캔 건수 누적기 초기화
    reference_date = source_start = source_end = None	# RFM 기준일 및 원본 날짜 범위 초기화
    for rows, dates, raw_start, raw_end, audit in transaction_chunks(	# 1차 스캔: 유효 고객·기간·RFM 기준일 수집
        transaction_path, start, end, sales_channel, chunk_size	# 청크 반복기 인자 전달
    ):
        counts.update(audit)	# 청크별 원본·제외 건수 누적
        if pd.notna(raw_start):	# 청크에 유효 날짜가 있는 경우
            source_start = raw_start if source_start is None else min(source_start, raw_start)	# 전체 원본 최소 날짜 갱신
            source_end = raw_end if source_end is None else max(source_end, raw_end)	# 전체 원본 최대 날짜 갱신
        if not rows.empty:	# 범위 내 유효 거래가 있는 경우
            customers.update(rows["customer_id"].unique())	# 고유 고객 ID 추가
            covered_months.update(dates.dt.to_period("M").unique())	# 관측된 연-월 추가
            reference_date = dates.max() if reference_date is None else max(reference_date, dates.max())	# 마지막 유효 구매일(RFM 기준일) 갱신
            counts["eligible_rows"] += len(rows)	# 추출 후보 유효 거래 행 수 누적

    expected_months = set(pd.period_range(start, end - pd.Timedelta(days=1), freq="M"))	# 기대되는 완전한 월 집합 계산
    if (source_start is None or source_start > start or source_end < end - pd.Timedelta(days=1)	# 원본이 요청 기간을 모두 덮는지 확인
            or covered_months != expected_months):	# 모든 월에 유효 거래가 존재하는지 확인
        raise ValueError("The source and channel do not cover the configured complete months.")	# 기간 불충분 시 예외 발생
    chosen_customers = choose(customers, max_customers, seed)	# 고정 시드로 표본 고객 선정
    population_size = len(customers)	# 전체 유효 고객 모수 기록
    del customers	# 메모리 절약을 위해 전체 고객 집합 해제

    output_dir.mkdir(parents=True, exist_ok=True)	# 출력 디렉터리 생성
    destination = output_dir / "hm_sample.zip"	# 최종 ZIP 파일 경로 설정
    with TemporaryDirectory(dir=output_dir) as temp_dir:	# 출력 폴더 내 임시 작업 디렉터리 생성
        work = Path(temp_dir)	# 임시 경로를 Path 객체로 래핑
        sample_csv = work / "transactions_sample.csv"	# 추출된 거래 CSV 임시 경로
        transaction_count = 0	# 추출된 거래 행 수 카운터
        article_ids = set()	# 추출된 거래에 등장한 상품 ID 집합
        for rows, _, _, _, _ in transaction_chunks(transaction_path, start, end, sales_channel, chunk_size):	# 2차 스캔: 선정 고객 거래만 추출
            sample = rows.loc[rows["customer_id"].isin(chosen_customers)]	# 표본 고객에 해당하는 행 필터링
            if sample.empty:	# 해당 청크에 표본 고객 거래가 없으면 건너뜀
                continue	# 다음 청크로 이동
            sample.to_csv(sample_csv, mode="a", header=transaction_count == 0, index=False)	# 첫 쓰기만 헤더 포함해 CSV에 이어쓰기
            transaction_count += len(sample)	# 추출 거래 행 수 누적
            article_ids.update(sample["article_id"].dropna().unique())	# 연결된 고유 상품 ID 수집
        if transaction_count < min_transactions:	# 최소 거래 건수 미달 시 검사
            raise ValueError(f"Only {transaction_count} eligible sample rows. Increase MAX_CUSTOMERS or the period.")	# 표본 부족 안내 예외

        articles = pd.read_csv(dataset_dir / "articles.csv", usecols=ARTICLE_COLUMNS, dtype="string")	# 상품 메타데이터 필수 열만 읽기
        articles = articles.loc[articles["article_id"].isin(article_ids)].copy()	# 추출 거래에 포함된 상품만 필터링
        if articles["article_id"].duplicated().any():	# 상품 ID 중복 여부 검증
            raise ValueError("Duplicate product IDs in the selected article metadata.")	# 중복 상품 ID 발견 시 예외
        image_paths = {	# 상품 ID별 원본 이미지 파일 경로 매핑 생성
            article_id: dataset_dir / "images" / article_id[:3] / f"{article_id}.jpg"	# 앞 3자리 하위 폴더 규칙 경로 조합
            for article_id in articles["article_id"]	# 추출된 상품 ID 순회
            if article_id.isascii() and article_id.isdecimal()	# 경로 안전성을 위해 ASCII 숫자 ID만 허용
        }
        image_paths = {article_id: path for article_id, path in image_paths.items() if path.is_file()}	# 실제 디스크에 존재하는 이미지만 유지
        if not image_paths:	# 매칭된 이미지가 하나도 없으면 중단
            raise ValueError("No product images matched the selected transactions and metadata.")	# 이미지 매칭 실패 예외
        chosen_images = choose(image_paths, max_images, seed)	# 고정 시드로 최대 max_images개 이미지 선정
        articles["source_image_available"] = articles["article_id"].isin(image_paths)	# 원본 이미지 존재 여부 플래그 열 추가
        articles["image_sampled"] = articles["article_id"].isin(chosen_images)	# 표본 이미지 포함 여부 플래그 열 추가
        articles.to_csv(work / "articles_sample.csv", index=False)	# 추출 상품 메타데이터 CSV 저장

        counts["sampled_rows"] = transaction_count	# 최종 추출 거래 행 수 기록
        counts["sample_excluded_rows"] = counts["eligible_rows"] - transaction_count	# 표본 미선정 고객의 유효 거래 행 수 계산
        excluded = sum(counts[name] for name in (	# 모든 제외 행 수 합계 계산
            "missing_customer_rows", "invalid_date_rows", "invalid_amount_rows",	# 정제 단계 제외 항목
            "out_of_scope_rows", "sample_excluded_rows",	# 범위·샘플링 단계 제외 항목
        ))
        if counts["sample_excluded_rows"] < 0 or counts["source_rows"] != excluded + transaction_count:	# 원본 행 수와 (제외+추출) 합계 정합성 검증
            raise ValueError("Transaction counts could not be reconciled.")	# 건수 불일치 시 예외 발생
        manifest = {	# 표본 추출 감사 로그(manifest) 딕셔너리 구성
            "source": "h-and-m-personalized-fashion-recommendations",	# 원본 Kaggle 대회 식별자
            "source_start": str(source_start.date()),	# 원본 전체 시작일
            "source_end": str(source_end.date()),	# 원본 전체 종료일
            "start_inclusive": str(start.date()),	# 추출 시작일(포함)
            "end_exclusive": str(end.date()),	# 추출 종료일(미포함)
            "sales_channel": sales_channel,	# 판매 채널 필터 설정값
            "seed": seed,	# 추출에 사용한 난수 시드
            "reference_date": str(reference_date.date()),	# 계산된 RFM 기준일
            "eligible_customers": population_size,	# 기간 내 전체 유효 고객 수
            "sampled_customers": len(chosen_customers),	# 최종 추출된 고객 수
            "sampled_product_ids": len(article_ids),	# 추출 거래에 등장한 고유 상품 수
            "matched_products": len(articles),	# 상품 메타데이터와 매칭된 상품 수
            "matched_products_without_images": len(articles) - len(image_paths),	# 이미지 파일이 없는 상품 수
            "products_with_images_outside_image_sample": len(image_paths) - len(chosen_images),	# 이미지가 있으나 표본에 미포함된 상품 수
            "sampled_images": len(chosen_images),	# 최종 ZIP에 포함된 이미지 수
            "transactions": dict(counts),	# 거래 행 수 정합성 감사 내역
        }
        (work / "sample_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")	# 감사 로그 JSON 저장
        archive_path = work / destination.name	# 임시 ZIP 파일 경로 지정
        with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:	# ZIP 아카이브 생성(기본 Deflate 압축)
            for name in ("transactions_sample.csv", "articles_sample.csv", "sample_manifest.json"):	# CSV 및 JSON 파일 목록 순회
                archive.write(work / name, name)	# 메타·거래·상품 파일을 아카이브 루트에 추가
            for article_id in sorted(chosen_images):	# 정렬된 순서로 표본 이미지 순회
                path = image_paths[article_id]	# 원본 이미지 파일 경로 조회
                archive.write(path, f"images/{path.parent.name}/{path.name}", compress_type=ZIP_STORED)	# 이미 압축된 JPG는 무압축(STORED)으로 저장
        archive_path.replace(destination)	# 완성된 ZIP을 최종 출력 경로로 원자적 이동
    return destination, manifest	# ZIP 경로와 감사 기록 반환


### 3. 표본 추출 실행 및 결과 요약 확인 (`In [3]`)

* 설정된 파라미터를 바탕으로 `prepare_hm_sample()`을 실행하여 `/kaggle/working/hm_sample.zip` 아카이브를 생성합니다.
* 생성 완료 후 압축 파일 용량(`45.46 MiB`), 추출된 유효 거래 행 수(`8,084`건), 표본 고객 수(`1,000`명), 포함된 상품 이미지 수(`200`장), 그리고 RFM 기준일(`2020-08-31`)을 출력하여 무결성을 최종 확인합니다.

In [ ]:
dataset_dir = DATASET_DIR if DATASET_DIR is not None else find_dataset(INPUT_ROOT)	# 수동 경로가 없으면 자동 탐색으로 데이터셋 경로 결정
archive, manifest = prepare_hm_sample(	# 표본 추출 및 ZIP 생성 실행
    dataset_dir, OUTPUT_DIR,	# 입력 데이터셋 및 출력 경로 전달
    start=START_DATE, end=END_DATE, seed=SEED,	# 기간 및 난수 시드 전달
    max_customers=MAX_CUSTOMERS, max_images=MAX_IMAGES,	# 고객·이미지 최대 추출 수 전달
    min_transactions=MIN_TRANSACTIONS, chunk_size=CHUNK_SIZE,	# 최소 거래 수 및 청크 크기 전달
    sales_channel=SALES_CHANNEL,	# 판매 채널 필터 전달
)
print(f"Archive: {archive.name} ({archive.stat().st_size / (1024 ** 2):.2f} MiB)")	# 생성된 ZIP 파일명과 용량(MiB) 출력
print(f"Rows: {manifest['transactions']['sampled_rows']:,}")	# 추출된 유효 거래 행 수 출력
print(f"Customers: {manifest['sampled_customers']:,}")	# 추출된 고객 수 출력
print(f"Product images: {manifest['sampled_images']:,}")	# 포함된 상품 이미지 수 출력
print(f"RFM reference date: {manifest['reference_date']}")	# 산출된 RFM 기준일 출력
